In [ ]:
import numpy as np
import pandas as pd
import skimage
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
IMAGE_DIR = "/home/cyril/development/data/NoduLoCC2026"
IMG_SIZE = 512

In [ ]:
# Load CSV
df = pd.read_csv(f"{IMAGE_DIR}/classification_labels.csv")

filenames = df["file_name"].values
labels = df["label"].values
# print(np.unique(df["label"].values))
labels = (labels == 'Nodule').astype(np.float32)

In [ ]:
def load_image(path):
    image = Image.open(path)
    image = np.array(image)
    # Check non "classical" 1024x1024
    if image.shape != (1024, 1024):
        # RGBA case
        if image.shape == (1024, 1024, 4):
            if (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 0] == image[:, :, 2]).all() and np.all(image[:, :, 3] == 255):
                image = image[:, :, 0]
        # Non square case
        elif len(image.shape) == 2:
            # Pad
            if image.shape[0] > image.shape[1]:
                image = np.pad(image, ((0, 0), (0, image.shape[0] - image.shape[1])), mode='symmetric')
            elif image.shape[0] < image.shape[1]:
                image = np.pad(image, ((0, image.shape[1] - image.shape[0]), (0, 0)), mode='symmetric')
            # Check padding
            if not (image.shape[0] == image.shape[1]):
                raise ValueError(image.shape)
            # Resize
            image = skimage.transform.resize(image, (1024, 1024), anti_aliasing=True)
        else:
            raise ValueError("ERROR", image.shape, path)
    # Downscale
    image = image.reshape(512, 2, 512, 2).mean(axis=(1, 3)) # downscale // 2
    
    image = (image / 255.0).astype(np.float16)
    return image

In [ ]:
X = np.zeros((len(labels), IMG_SIZE, IMG_SIZE, 1), dtype=np.float16)
Y = np.zeros((len(labels), 1), dtype=np.float32)

for i in tqdm(range(len(labels))):
    image = load_image(f'{IMAGE_DIR}/nih_filtered_images/{filenames[i]}')
    label = labels[i]
    
    X[i, :, :, 0] = image
    Y[i, 0] = label

In [ ]:
np.save('X.npy', X)
np.save('Y.npy', Y)